In [24]:
import math
import numpy as np
from matplotlib.ticker import EngFormatter



# Python functs for values calculation

### Bulk Capacitor

In [ ]:
def calc_cin(P_IN, V_BULK_min, V_IN_min, f_LINE_min): # C_bulk
    return (2 * P_IN * (0.25 + (1 / math.pi) * math.asin(V_BULK_min / (math.sqrt(2) * V_IN_min)))) / ((2 * V_IN_min**2 - V_BULK_min**2) * f_LINE_min)

In [26]:
# Define the parameters
P_IN = 105 # Input power in watts
V_IN_min = 110  # Minimum input voltage in volts RMS

# Minimum bulk voltage in volts DC/prom --> Realmente toca diseñar primero otras etapas para definir este valor,
#pero se puede usar un valor estimado para asegurar que el bulk voltage no caiga por debajo de un cierto nivel durante la operación. 
# En este caso, se ha considerado una caída de 2 diodos (2*0.7V) y un margen de seguridad del 10% (0.9) para obtener un valor estimado del 
# voltaje mínimo del bulk.
V_BULK_min = (np.sqrt(2)*V_IN_min-2*0.7)*0.9   
f_LINE_min = 58  # Minimum line frequency in Hz

# Calculate the required bulk capacitance
C_bulk = calc_cin(P_IN, V_BULK_min, V_IN_min, f_LINE_min)
print(EngFormatter(unit='F').format_data(C_bulk))

439.399 µF


### Transformer turns ratio and Maximun Duty Cycle


In [32]:
def calc_vbulk_max(V_IN_max):
    return np.sqrt(2) * V_IN_max

def calc_vreflected(V_DS_rated, V_BULK_max):
    return 0.8 * (V_DS_rated - 1.3 * V_BULK_max)

def calc_nps(V_REFLECTED, V_OUT):
    return V_REFLECTED / V_OUT

def calc_npa(N_PS, V_OUT, V_BIAS):
    return N_PS * (V_OUT / V_BIAS)

def calc_vdiode(V_BULK_max, N_PS, V_OUT):
    return (V_BULK_max / N_PS) + V_OUT

def calc_dmax(N_PS, V_OUT, V_F, V_BULK_min):
    return (N_PS * (V_OUT + V_F)) / (V_BULK_min + N_PS * (V_OUT + V_F))


In [ ]:
# Define parameters
V_IN_max = 130  # Maximum input voltage in volts RMS
Vbulk_max = calc_vbulk_max(V_IN_max)

V_DS_rated = 600  # Rated drain-source voltage of the MOSFET in volts
V_reflected = calc_vreflected(V_DS_rated, Vbulk_max)

Vo = 35  # Output voltage in volts
Nps = calc_nps(V_reflected, Vo)  

print(f"Maximum Bulk Voltage: {EngFormatter(unit='V').format_data(Vbulk_max)}")
print(f"Reflected Voltage: {EngFormatter(unit='V').format_data(V_reflected)}")
print(f"Number of Primary Turns (Nps): {Nps:.2f}")



Maximum Bulk Voltage: 183.848 V
Reflected Voltage: 288.798 V
Number of Primary Turns (Nps): 8.25


In [33]:
# Define parameters
Vbias = 15  # Bias voltage in volts
VF_diode = 0.7  # Forward voltage drop of the diode in volts

npa = calc_npa(Nps, Vo, Vbias)  
vdiode = calc_vdiode(Vbulk_max, Nps, Vo)
D = calc_dmax(Nps, Vo, VF_diode, V_BULK_min)


print(f"Number of Primary Turns (Npa): {npa:.2f}")
print(f"Diode Voltage: {EngFormatter(unit='V').format_data(vdiode)}")
print(f"Maximum Duty Cycle (Dmax): {D:.2f}")


Number of Primary Turns (Npa): 19.25
Diode Voltage: 57.2808 V
Maximum Duty Cycle (Dmax): 0.68


### InductanceP

In [35]:
def calc_lp(V_BULK_min, N_PS, V_OUT, P_IN, f_SW):
    return (0.5 * (V_BULK_min**2) * ((N_PS * V_OUT) / (V_BULK_min + N_PS * V_OUT))**2) / (0.1 * P_IN * f_SW)

def calc_ipk_mosfet(P_IN, V_BULK_min, N_PS, V_OUT, L_m, f_SW):
    return (
        P_IN /
        (
            V_BULK_min *
            (
                (N_PS * V_OUT) /
                (V_BULK_min + (N_PS * V_OUT))
            )
        )
    ) + (
        (V_BULK_min / (2 * L_m)) *
        (
            (
                (N_PS * V_OUT) /
                (V_BULK_min + (N_PS * V_OUT))
            ) / f_SW
        )
    )

def calc_irms_mosfet(D_MAX, V_BULK_min, L_P, f_SW, I_PK_MOSFET):
    return np.sqrt(
        ((D_MAX**3) / 3) *
        ((V_BULK_min / (L_P * f_SW))**2)
        -
        (
            (D_MAX**2 * I_PK_MOSFET * V_BULK_min) /
            (L_P * f_SW)
        )
        +
        (
            D_MAX * (I_PK_MOSFET**2)
        )
    )

def calc_ipk_diode(N_PS, I_PK_MOSFET):
    return N_PS * I_PK_MOSFET

In [39]:
# Define parameters
f_sw = 300e3  # Switching frequency in hertz
lp = calc_lp(V_BULK_min, Nps, Vo, P_IN, f_sw)
ipk = calc_ipk_mosfet(P_IN, V_BULK_min, Nps, Vo, lp, f_sw)
irms = calc_irms_mosfet(D, V_BULK_min, lp, f_sw, ipk)
ipk_diode = calc_ipk_diode(Nps, ipk)

print(f"Primary Inductance (Lp): {EngFormatter(unit='H').format_data(lp)}")
print(f"Peak Current through MOSFET (Ipk): {EngFormatter(unit='A').format_data(ipk)}")  
print(f"RMS Current through MOSFET (Irms): {EngFormatter(unit='A').format_data(irms)}") 
print(f"Peak Current through Diode (Ipk_diode): {EngFormatter(unit='A').format_data(ipk_diode)}")


Primary Inductance (Lp): 1.39422 mH
Peak Current through MOSFET (Ipk): 1.23238 A
RMS Current through MOSFET (Irms): 924.698 mA
Peak Current through Diode (Ipk_diode): 10.1689 A


### C out

In [ ]:
def calc_cout(I_OUT, N_PS, V_OUT, V_BULK_min, f_SW,percent_ripple):
    return (
        I_OUT *
        (
            (N_PS * V_OUT) /
            (V_BULK_min + N_PS * V_OUT)
        )
    ) / (
        percent_ripple * V_OUT * f_SW
    )

In [42]:
# Define the parameters
percent_ripple = 0.001  # Desired ripple percentage (1%)
I_OUT = 3  # Output current in amperes

c_out = calc_cout(I_OUT, Nps, Vo, V_BULK_min, f_sw, percent_ripple)

print(f"Output Capacitance (Cout): {EngFormatter(unit='F').format_data(c_out)}")

Output Capacitance (Cout): 192.994 µF
